# Building neural networks with Keras, step by step

This notebook takes you from a single neuron to a trained image classifier, adding one idea at a time and plotting what changes at every stage.

**How it is organised**

| Part | What you build | What you learn to see |
|---|---|---|
| 0 | Setup and a 2-D toy dataset | Why a toy dataset is the right place to start |
| 1 | One neuron | The `Sequential` / `compile` / `fit` loop |
| 2 | One hidden layer | What non-linearity buys you |
| 3 | A sweep over depths and widths | Capacity, and when more stops helping |
| 4 | Activation comparison | How `relu`, `tanh` and `sigmoid` train differently |
| 5 | Learning rate, optimizer, batch size | Reading loss curves for instability |
| 6 | Overfitting, then four fixes | The gap between training and validation |
| 7 | Callbacks and schedules | Letting training manage itself |
| 8 | MNIST digits | A real multi-class problem end to end |
| 9 | Functional API, saving, loading | Models beyond a simple stack |

Every section follows the same rhythm: build, train, **plot the curves**, then read what the plot is telling you.

**Run it in order.** Later cells reuse helper functions defined earlier.

## How to run this

You need Python 3.9+ and these packages:

```bash
pip install tensorflow matplotlib numpy scikit-learn
```

On Apple Silicon use `pip install tensorflow` as well (it ships with Metal support in recent versions). Google Colab already has everything installed, so you can upload this notebook and run it there with no setup.

Everything here runs comfortably on a CPU — expect roughly 5 to 10 minutes for the whole notebook on a laptop, most of it in Parts 5 and 6 where several models are trained for comparison. If you are impatient, lower the `epochs` arguments; the shape of the curves survives.

In [ ]:
# Core imports. Run this cell first.
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU") or "none (CPU is fine for this notebook)")

# Make plots readable and consistent.
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

In [ ]:
# Reproducibility: seeds the Python, NumPy and backend random number generators at once.
# Training is still not bit-for-bit deterministic on GPU, but results become stable enough to compare.
SEED = 42
keras.utils.set_random_seed(SEED)

## Part 0 — A dataset you can see

Before touching a real dataset, use one you can plot on a single pair of axes. With two input features, you can draw the network's entire decision surface, which makes every change you make visible instead of abstract.

`make_moons` produces two interleaving half-circles. They are not separable by a straight line, which is exactly the point: it will expose the difference between a network with and without a hidden layer.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=1000, noise=0.25, random_state=SEED)

# Three-way split: train to fit weights, validation to watch during training, test for one final honest number.
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=SEED, stratify=y_temp)

# Standardise features using statistics from the TRAINING set only.
# Fitting the scaler on all the data would leak information about validation and test into training.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

print(f"train {X_train.shape}  val {X_val.shape}  test {X_test.shape}")
print("class balance in train:", np.bincount(y_train))

In [ ]:
def plot_data(ax=None, title="The two moons"):
    # Scatter the training points, coloured by class.
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(*X_train[y_train == 0].T, s=14, c="#B86A0C", label="class 0", edgecolors="white", linewidths=0.4)
    ax.scatter(*X_train[y_train == 1].T, s=14, c="#2447D6", label="class 1", edgecolors="white", linewidths=0.4)
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.legend(loc="upper right")
    return ax

plot_data()
plt.show()

Ask yourself before modelling: *could I separate these with one straight line?* No. Keep that answer in mind for Part 1.

## Part 1 — One neuron

The smallest useful Keras model is a single `Dense` unit with a sigmoid activation. It computes

```
z = w1*x1 + w2*x2 + b
ŷ = sigmoid(z)
```

which is logistic regression. It is worth building, because the three Keras steps you use here are the same three you will use for a 100-layer model.

**1. Define the architecture**

```python
model = keras.Sequential([keras.Input(shape=(2,)), layers.Dense(1, activation="sigmoid")])
```

`Sequential` is a straight stack of layers. `keras.Input` declares the shape of one example (no batch dimension), which lets Keras build the weights immediately so `summary()` works.

**2. Compile: choose the loss, the optimizer and what to report**

```python
model.compile(optimizer=..., loss=..., metrics=[...])
```

- **loss** is what the optimizer minimises. `binary_crossentropy` for a two-class problem with one sigmoid output.
- **optimizer** is the update rule. `Adam` is the sane default.
- **metrics** are numbers reported for your benefit only; they are not optimised.

**3. Fit: run the training loop**

```python
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=..., batch_size=...)
```

`fit` returns a `History` object, and `history.history` is a dictionary of lists, one value per epoch. That dictionary is the raw material for every curve in this notebook.

In [ ]:
keras.utils.set_random_seed(SEED)

model_1 = keras.Sequential([
    keras.Input(shape=(2,)),            # two features in
    layers.Dense(1, activation="sigmoid")  # one neuron out
], name="single_neuron")

model_1.summary()

Read the summary: **3 parameters**. Two weights (one per input) and one bias — exactly the `w1, w2, b` in the formula above. Whenever you are unsure what a layer is doing, count its parameters.

In [ ]:
model_1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_1 = model_1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    verbose=0,          # set to 1 to watch the per-epoch progress bars
)

print("keys recorded by fit():", list(history_1.history.keys()))
print(f"final train accuracy: {history_1.history['accuracy'][-1]:.3f}")
print(f"final val   accuracy: {history_1.history['val_accuracy'][-1]:.3f}")

### The plotting helper you will reuse

Every remaining section calls this. It accepts either one `History` or a dictionary of named histories, so the same function draws a single run or a comparison of many.

In [ ]:
def plot_history(histories, metrics=("loss", "accuracy"), title=None, figsize=(11, 4)):
    # histories: a single History, or a dict {name: History} to overlay several runs.
    if not isinstance(histories, dict):
        histories = {"model": histories}

    fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
    axes = np.atleast_1d(axes)
    colors = plt.cm.viridis(np.linspace(0, 0.85, max(len(histories), 1)))

    for (name, h), color in zip(histories.items(), colors):
        hist = h.history if hasattr(h, "history") else h
        for ax, metric in zip(axes, metrics):
            if metric not in hist:
                continue
            epochs = np.arange(1, len(hist[metric]) + 1)
            ax.plot(epochs, hist[metric], color=color, label=f"{name} (train)")
            if "val_" + metric in hist:
                # Dashed = validation. Same colour as its training twin, so pairs are easy to read.
                ax.plot(epochs, hist["val_" + metric], color=color, linestyle="--", label=f"{name} (val)")

    for ax, metric in zip(axes, metrics):
        ax.set_xlabel("epoch")
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend(fontsize=8)

    if title:
        fig.suptitle(title, y=1.02, fontsize=13)
    fig.tight_layout()
    return fig


plot_history(history_1, title="Single neuron: solid = training, dashed = validation")
plt.show()

### The decision boundary helper

Loss curves tell you *how well*; the decision surface tells you *what the model actually learned*. For a 2-D input you can colour every point of the plane by the model's predicted probability.

In [ ]:
def plot_boundary(model, ax=None, title=None, resolution=220):
    # Colour the plane by predicted probability, then overlay the training points.
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5))

    pad = 0.6
    x_min, x_max = X_train[:, 0].min() - pad, X_train[:, 0].max() + pad
    y_min, y_max = X_train[:, 1].min() - pad, X_train[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))

    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    probs = model.predict(grid, verbose=0).reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=25, cmap="coolwarm_r", alpha=0.75, vmin=0, vmax=1)
    ax.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=1.4)  # the 50% line
    ax.scatter(*X_train[y_train == 0].T, s=10, c="#7a4408", edgecolors="white", linewidths=0.3)
    ax.scatter(*X_train[y_train == 1].T, s=10, c="#13246e", edgecolors="white", linewidths=0.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if title:
        ax.set_title(title, fontsize=11)
    return ax


plot_boundary(model_1, title="One neuron: the boundary is a straight line")
plt.show()

**What the plot says.** The black line is where the model predicts 50%. With one neuron it is perfectly straight, and no amount of extra training will bend it — a single weighted sum can only ever split the plane with a hyperplane. Validation accuracy plateaus around 0.82 and stops there no matter how long you train.

This is the concrete reason hidden layers exist. Next.

## Part 2 — Add a hidden layer

Insert eight neurons between the input and the output, with a `relu` activation. The output layer stays the same.

```
input (2) → Dense(8, relu) → Dense(1, sigmoid)
```

Two things to notice in the code: the hidden layer's activation is **not** optional (without it the two `Dense` layers collapse into one linear map), and the parameter count jumps from 3 to 33.

In [ ]:
keras.utils.set_random_seed(SEED)

model_2 = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),     # hidden layer: 2*8 weights + 8 biases = 24 params
    layers.Dense(1, activation="sigmoid"),  # output layer: 8*1 weights + 1 bias   =  9 params
], name="one_hidden_layer")

model_2.summary()

model_2.compile(optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=["accuracy"])
history_2 = model_2.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=100, batch_size=32, verbose=0)

print(f"val accuracy — 1 neuron: {history_1.history['val_accuracy'][-1]:.3f}"
      f"   |   8 hidden units: {history_2.history['val_accuracy'][-1]:.3f}")

In [ ]:
plot_history({"no hidden layer": history_1, "8 hidden units": history_2},
             title="Adding one hidden layer")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_boundary(model_1, axes[0], "No hidden layer")
plot_boundary(model_2, axes[1], "One hidden layer (8 relu units)")
plt.tight_layout(); plt.show()

The boundary is now **bent**. Each hidden relu unit contributes one fold, and the output layer combines them. That is the whole mechanism by which depth creates flexibility: every hidden unit adds a kink, and layers compose kinks into curves.

## Part 3 — Iterative building: sweep the architecture

Rather than guessing, write one function that builds and trains any stack, then loop over a list of designs. This is how architecture choices are actually made in practice: change one axis at a time and compare curves.

In [ ]:
def build_mlp(hidden=(8,), activation="relu", lr=1e-2, dropout=0.0, l2=0.0, input_shape=(2,)):
    # hidden: a tuple giving the width of each hidden layer, e.g. (16, 16) for two layers of 16.
    reg = keras.regularizers.l2(l2) if l2 else None
    model = keras.Sequential([keras.Input(shape=input_shape)])
    for width in hidden:
        model.add(layers.Dense(width, activation=activation, kernel_regularizer=reg))
        if dropout:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model


def train(model, epochs=150, batch_size=32, callbacks=None,
          data=None, verbose=0):
    Xtr, ytr, Xv, yv = data if data else (X_train, y_train, X_val, y_val)
    return model.fit(Xtr, ytr, validation_data=(Xv, yv), epochs=epochs,
                     batch_size=batch_size, callbacks=callbacks, verbose=verbose)

In [ ]:
architectures = {
    "none":        (),             # linear model, for reference
    "4":           (4,),
    "16":          (16,),
    "64":          (64,),
    "16-16":       (16, 16),
    "32-32-32":    (32, 32, 32),
}

results, models = {}, {}
for name, hidden in architectures.items():
    keras.utils.set_random_seed(SEED)              # same starting conditions for every run
    m = build_mlp(hidden=hidden)
    results[name] = train(m, epochs=120)
    models[name] = m
    print(f"{name:>10s}  params={m.count_params():>5d}  "
          f"train acc={results[name].history['accuracy'][-1]:.3f}  "
          f"val acc={results[name].history['val_accuracy'][-1]:.3f}")

In [ ]:
plot_history(results, title="Capacity sweep: wider and deeper", figsize=(12, 4.5))
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for ax, (name, m) in zip(axes.ravel(), models.items()):
    acc = results[name].history["val_accuracy"][-1]
    plot_boundary(m, ax, f"hidden = {name}   (val {acc:.3f})")
fig.suptitle("The same data, six capacities", fontsize=14)
plt.tight_layout(); plt.show()

**How to read this grid.** Moving from `none` to `16` buys a large jump. Moving from `16` to `32-32-32` buys very little accuracy but produces a visibly more contorted boundary that chases individual noisy points. That contortion is overfitting beginning to appear, and it is the reason you will not simply keep adding layers.

A practical rule: grow capacity until validation loss stops improving, then stop and spend your effort on regularisation and data instead.

## Part 4 — Activation functions, measured

Swap only the hidden activation and keep everything else fixed. Watch the loss curves rather than the final number: the shape of the early epochs is where activations differ most.

In [ ]:
act_results = {}
for act in ["relu", "tanh", "sigmoid", "elu"]:
    keras.utils.set_random_seed(SEED)
    m = build_mlp(hidden=(16, 16), activation=act, lr=5e-3)
    act_results[act] = train(m, epochs=120)
    print(f"{act:>8s}  val acc={act_results[act].history['val_accuracy'][-1]:.3f}")

plot_history(act_results, title="Hidden activation, everything else fixed", figsize=(12, 4.5))
plt.show()

Typical outcome: `relu` and `tanh` drop quickly, `sigmoid` lags because its slope is at most 0.25, so gradients shrink as they pass back through each sigmoid layer. With only two hidden layers the effect is mild; at ten layers sigmoid often fails to train at all. That is the vanishing-gradient problem, and it is why `relu` became the default.

Try rerunning this cell with `hidden=(16, 16, 16, 16, 16, 16)` to see the gap widen.

## Part 5 — Learning rate, optimizer, batch size

The learning rate is the single most important hyperparameter. These curves show what each failure mode looks like, so you can recognise it in your own training logs.

In [ ]:
lr_results = {}
for lr in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    keras.utils.set_random_seed(SEED)
    m = build_mlp(hidden=(16, 16), lr=lr)
    lr_results[f"lr={lr:g}"] = train(m, epochs=100)

plot_history(lr_results, metrics=("loss",), title="Learning rate sweep", figsize=(7.5, 4.5))
plt.ylim(0, 1.2)
plt.show()

Four signatures to memorise:

- **Too small** (1e-4): the loss falls, but far too slowly. Flat and still descending at the end.
- **About right** (1e-3 to 1e-2): a fast drop, then a smooth flattening.
- **Too large** (1e-1): a jagged curve that falls and jumps back up.
- **Far too large** (1.0): the loss stalls high or becomes `nan`. The steps overshoot the valley entirely.

When a model will not train, halve the learning rate before you change anything else about the architecture.

In [ ]:
opt_results = {}
optimizers = {
    "SGD":          keras.optimizers.SGD(0.05),
    "SGD+momentum": keras.optimizers.SGD(0.05, momentum=0.9),
    "RMSprop":      keras.optimizers.RMSprop(0.005),
    "Adam":         keras.optimizers.Adam(0.005),
}
for name, opt in optimizers.items():
    keras.utils.set_random_seed(SEED)
    m = keras.Sequential([keras.Input(shape=(2,)),
                          layers.Dense(16, activation="relu"),
                          layers.Dense(16, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    opt_results[name] = train(m, epochs=100)

plot_history(opt_results, title="Optimizers at a comparable learning rate", figsize=(12, 4.5))
plt.show()

In [ ]:
batch_results = {}
for bs in [8, 32, 128, len(X_train)]:
    keras.utils.set_random_seed(SEED)
    m = build_mlp(hidden=(16, 16), lr=5e-3)
    label = "full batch" if bs == len(X_train) else f"batch={bs}"
    batch_results[label] = train(m, epochs=80, batch_size=bs)

plot_history(batch_results, metrics=("loss",), title="Batch size", figsize=(7.5, 4.5))
plt.show()

Smaller batches mean more weight updates per epoch, so progress per epoch is faster — but each update is noisier, and the wall-clock time per epoch is worse because the hardware is used less efficiently. Full-batch training gives the smoothest curve and the slowest learning. 32 to 256 is the usual compromise.

## Part 6 — Overfitting, and four ways to fight it

To make overfitting unmistakable, shrink the training set to 60 points and use an oversized network. Watch the two curves separate.

In [ ]:
# A deliberately difficult setup: tiny training set, large model.
X_small, y_small = X_train[:60], y_train[:60]
small_data = (X_small, y_small, X_val, y_val)

keras.utils.set_random_seed(SEED)
overfit_model = build_mlp(hidden=(256, 256, 256), lr=1e-3)
hist_overfit = train(overfit_model, epochs=300, batch_size=16, data=small_data)

plot_history(hist_overfit, title="Overfitting: training loss falls, validation loss turns around")
plt.show()

best_epoch = int(np.argmin(hist_overfit.history["val_loss"])) + 1
print(f"validation loss was lowest at epoch {best_epoch} of {len(hist_overfit.history['val_loss'])}")
print(f"train acc {hist_overfit.history['accuracy'][-1]:.3f}  vs  val acc {hist_overfit.history['val_accuracy'][-1]:.3f}")

The dashed validation loss reaches a minimum and then climbs while the solid training loss keeps falling. Everything after that minimum is the model memorising the 60 training points, including their noise.

Now apply four standard remedies to the identical setup.

In [ ]:
fixes = {"no regularisation": hist_overfit}
fix_models = {"no regularisation": overfit_model}

# 1. Early stopping: halt when validation loss stops improving, and restore the best weights.
keras.utils.set_random_seed(SEED)
m = build_mlp(hidden=(256, 256, 256), lr=1e-3)
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
fixes["early stopping"] = train(m, epochs=300, batch_size=16, data=small_data, callbacks=[es])
fix_models["early stopping"] = m
print("early stopping halted after", len(fixes["early stopping"].history["loss"]), "epochs")

# 2. Dropout: randomly zero 40% of each hidden layer's units during training.
keras.utils.set_random_seed(SEED)
m = build_mlp(hidden=(256, 256, 256), lr=1e-3, dropout=0.4)
fixes["dropout 0.4"] = train(m, epochs=300, batch_size=16, data=small_data)
fix_models["dropout 0.4"] = m

# 3. L2 weight decay: penalise large weights so the fit stays smooth.
keras.utils.set_random_seed(SEED)
m = build_mlp(hidden=(256, 256, 256), lr=1e-3, l2=1e-2)
fixes["L2 = 0.01"] = train(m, epochs=300, batch_size=16, data=small_data)
fix_models["L2 = 0.01"] = m

# 4. A smaller model: less capacity to memorise in the first place.
keras.utils.set_random_seed(SEED)
m = build_mlp(hidden=(8,), lr=1e-3)
fixes["small model (8)"] = train(m, epochs=300, batch_size=16, data=small_data)
fix_models["small model (8)"] = m

for name, h in fixes.items():
    print(f"{name:>20s}  best val loss={min(h.history['val_loss']):.3f}  "
          f"final val acc={h.history['val_accuracy'][-1]:.3f}")

In [ ]:
plot_history(fixes, metrics=("val_loss", "val_accuracy"),
             title="Four remedies, same data and same oversized network", figsize=(12, 4.5))
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, m) in zip(axes, fix_models.items()):
    plot_boundary(m, ax, name)
plt.tight_layout(); plt.show()

Compare the boundaries. The unregularised model draws tight islands around individual points; the regularised ones draw a smooth curve that generalises. The fifth remedy, not shown here because it is not a code trick, is **more data**: refit any of these on the full 640-point training set and the overfitting largely disappears.

## Part 7 — Callbacks: let training manage itself

Callbacks are objects that Keras calls at defined moments during `fit`. The four you will use constantly:

| Callback | Job |
|---|---|
| `EarlyStopping` | Stop when a monitored metric stops improving; restore the best weights |
| `ModelCheckpoint` | Save the best model to disk as training proceeds |
| `ReduceLROnPlateau` | Cut the learning rate when progress stalls |
| `CSVLogger` | Write per-epoch metrics to a file |

The cell below also records the learning rate each epoch with a tiny custom callback, so you can plot the schedule alongside the loss.

In [ ]:
class LRLogger(keras.callbacks.Callback):
    # A minimal custom callback: append the current learning rate to the history each epoch.
    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        logs["lr"] = float(keras.ops.convert_to_numpy(lr)) if hasattr(keras, "ops") else float(lr.numpy())


keras.utils.set_random_seed(SEED)
model_cb = build_mlp(hidden=(32, 32), lr=1e-2)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=40, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=12, min_lr=1e-5, verbose=0),
    keras.callbacks.ModelCheckpoint("best_moons_model.keras", monitor="val_loss", save_best_only=True),
    LRLogger(),
]

hist_cb = train(model_cb, epochs=400, callbacks=callbacks)
print("stopped after", len(hist_cb.history["loss"]), "epochs")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_cb.history["loss"], label="train")
axes[0].plot(hist_cb.history["val_loss"], "--", label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist_cb.history["lr"], color="#B86A0C")
axes[1].set_yscale("log")
axes[1].set_title("learning rate (reduced on plateau)"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

Each downward step in the right-hand plot is `ReduceLROnPlateau` reacting to a stalled validation loss. Notice how the loss on the left usually takes another small drop just after each cut — large steps get you near the valley, small steps settle into it.

In [ ]:
# The honest number: evaluate once, on data the model has never seen and that you never tuned against.
best = keras.models.load_model("best_moons_model.keras")
test_loss, test_acc = best.evaluate(X_test, y_test, verbose=0)
print(f"test loss {test_loss:.3f}   test accuracy {test_acc:.3f}")

## Part 8 — A real problem: handwritten digits

Same four steps, bigger data. Now there are ten classes, so three things change:

- the output layer has **10 units with `softmax`** instead of 1 with `sigmoid`;
- the loss becomes `sparse_categorical_crossentropy` (use the `sparse` version when labels are integers like `7`; use plain `categorical_crossentropy` when they are one-hot vectors);
- the images need flattening from a 2-D grid into a vector, done by a `Flatten` layer.

In [ ]:
# MNIST downloads about 11 MB the first time. If there is no network, fall back to scikit-learn's
# bundled 8x8 digits so the rest of the notebook still runs.
try:
    (x_tr_full, y_tr_full), (x_te, y_te) = keras.datasets.mnist.load_data()
    DATASET = "MNIST (28x28)"
except Exception as err:
    print("Could not download MNIST, falling back to sklearn digits. Reason:", err)
    from sklearn.datasets import load_digits
    d = load_digits()
    imgs = d.images.astype("float32") / 16.0 * 255.0
    x_tr_full, x_te, y_tr_full, y_te = train_test_split(
        imgs, d.target, test_size=0.2, random_state=SEED, stratify=d.target)
    DATASET = "sklearn digits (8x8)"

# Scale pixels to [0, 1]. Networks train far better on small, similarly scaled inputs.
x_tr_full = x_tr_full.astype("float32") / 255.0
x_te = x_te.astype("float32") / 255.0

# Carve a validation set out of the training data.
x_tr, x_va, y_tr, y_va = train_test_split(x_tr_full, y_tr_full, test_size=0.1,
                                          random_state=SEED, stratify=y_tr_full)

print(DATASET)
print(f"train {x_tr.shape}   val {x_va.shape}   test {x_te.shape}")
print("pixel range:", x_tr.min(), "to", x_tr.max())

In [ ]:
# Always look at the data before modelling it.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, label in zip(axes.ravel(), x_tr, y_tr):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
plt.suptitle("Sixteen training examples", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
keras.utils.set_random_seed(SEED)

mnist_model = keras.Sequential([
    keras.Input(shape=x_tr.shape[1:]),       # (28, 28) for MNIST
    layers.Flatten(),                        # 28*28 = 784 numbers, no parameters of its own
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),  # ten probabilities that sum to 1
], name="mnist_mlp")

mnist_model.summary()

mnist_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",  # integer labels
    metrics=["accuracy"],
)

In [ ]:
mnist_cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
]

hist_mnist = mnist_model.fit(
    x_tr, y_tr,
    validation_data=(x_va, y_va),
    epochs=30,
    batch_size=128,
    callbacks=mnist_cbs,
    verbose=1,     # worth watching live: this is the run that takes the longest
)

In [ ]:
plot_history(hist_mnist, title=f"Dense network on {DATASET}")
plt.show()

test_loss, test_acc = mnist_model.evaluate(x_te, y_te, verbose=0)
print(f"test accuracy: {test_acc:.4f}   test loss: {test_loss:.4f}")

### Where are the mistakes?

Aggregate accuracy hides structure. A confusion matrix shows which classes the model mixes up, and looking at the actual misclassified images usually explains why.

In [ ]:
probs = mnist_model.predict(x_te, verbose=0)   # shape (n_samples, 10)
preds = probs.argmax(axis=1)

cm = np.zeros((10, 10), dtype=int)
for true, pred in zip(y_te, preds):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Confusion matrix")
ax.grid(False)
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max() * 0.5 else "black")
plt.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.show()

# Per-class accuracy, worst first.
per_class = cm.diagonal() / cm.sum(axis=1)
for digit in np.argsort(per_class):
    print(f"digit {digit}: {per_class[digit]:.3f}")

In [ ]:
wrong = np.flatnonzero(preds != y_te)
print(f"{len(wrong)} wrong out of {len(y_te)}")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for ax, idx in zip(axes.ravel(), wrong[:16]):
    ax.imshow(x_te[idx], cmap="gray_r")
    ax.set_title(f"saw {preds[idx]}, was {y_te[idx]}\n{probs[idx].max():.0%} sure", fontsize=8)
    ax.axis("off")
plt.suptitle("Where it went wrong", y=1.06)
plt.tight_layout(); plt.show()

Look for confident mistakes (high percentage, wrong answer). Those are the ones worth investigating: they often reveal genuinely ambiguous handwriting, or a systematic weakness such as confusing 4 and 9.

### Iterating on the image model

Apply the same three-step loop you used on the moons: add capacity, add regularisation, compare curves. `BatchNormalization` is new here — it rescales the activations inside the network, which stabilises and usually speeds up training.

In [ ]:
def build_mnist(hidden=(128, 64), dropout=0.0, batchnorm=False, lr=1e-3):
    model = keras.Sequential([keras.Input(shape=x_tr.shape[1:]), layers.Flatten()])
    for width in hidden:
        model.add(layers.Dense(width, use_bias=not batchnorm))
        if batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))
        if dropout:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


variants = {
    "128-64":                   dict(hidden=(128, 64)),
    "512-256-128":              dict(hidden=(512, 256, 128)),
    "512-256-128 + dropout 0.3": dict(hidden=(512, 256, 128), dropout=0.3),
    "512-256-128 + BN + drop":   dict(hidden=(512, 256, 128), dropout=0.3, batchnorm=True),
}

mnist_results = {}
for name, kwargs in variants.items():
    keras.utils.set_random_seed(SEED)
    m = build_mnist(**kwargs)
    mnist_results[name] = m.fit(x_tr, y_tr, validation_data=(x_va, y_va),
                                epochs=12, batch_size=128, verbose=0)
    acc = m.evaluate(x_te, y_te, verbose=0)[1]
    print(f"{name:>28s}  params={m.count_params():>8,d}  test acc={acc:.4f}")

plot_history(mnist_results, title="Iterating on the digit classifier", figsize=(12, 4.5))
plt.show()

Typically the bigger plain network reaches lower *training* loss but not better *validation* accuracy, and dropout closes the gap. Around 98% is roughly the ceiling for dense layers on MNIST; getting beyond it needs convolutional layers, which exploit the fact that pixels near each other are related — a natural next notebook.

## Part 9 — Beyond `Sequential`

`Sequential` only handles a straight stack. The **functional API** lets you build any graph: multiple inputs, multiple outputs, branches, and skip connections. You call each layer on a tensor and wire the pieces together yourself.

In [ ]:
# The same classifier, written functionally, plus a skip connection that Sequential cannot express.
inputs = keras.Input(shape=x_tr.shape[1:], name="image")
x = layers.Flatten()(inputs)
x = layers.Dense(128, activation="relu")(x)

branch = layers.Dense(128, activation="relu")(x)
x = layers.Add()([x, branch])          # skip connection: add the input of the branch back to its output

x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax", name="digit")(x)

func_model = keras.Model(inputs, outputs, name="mlp_with_skip")
func_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
func_model.summary()

In [ ]:
hist_func = func_model.fit(x_tr, y_tr, validation_data=(x_va, y_va),
                           epochs=8, batch_size=128, verbose=0)
print("test accuracy:", f"{func_model.evaluate(x_te, y_te, verbose=0)[1]:.4f}")
plot_history(hist_func, title="Functional model with a skip connection")
plt.show()

In [ ]:
# Saving and reloading. The .keras format stores architecture, weights and optimizer state in one file.
mnist_model.save("digit_classifier.keras")
reloaded = keras.models.load_model("digit_classifier.keras")

print("same predictions after reload:",
      np.allclose(mnist_model.predict(x_te[:50], verbose=0),
                  reloaded.predict(x_te[:50], verbose=0)))

# Predicting on a single example: the model always expects a batch, so add a leading axis.
single = x_te[0][np.newaxis, ...]
p = reloaded.predict(single, verbose=0)[0]
print(f"predicted {p.argmax()} with {p.max():.1%} confidence; true label {y_te[0]}")

## Recap

The whole workflow, in the order you will always run it:

1. **Look at the data.** Plot it. Split it into train / validation / test before anything else. Scale the inputs.
2. **Start too small.** A linear model or one small hidden layer gives you a baseline to beat.
3. **Build.** `Sequential` for a stack, functional API for anything branching. Hidden layers get `relu`; the output layer's activation is dictated by the task.
4. **Compile.** Loss matched to the task, `Adam(1e-3)` as a starting optimizer, metrics for your own reading.
5. **Fit, and keep the `History`.** Every diagnosis in this notebook came from those curves.
6. **Read the curves.**
   - Both losses still falling → train longer, or raise capacity.
   - Training loss falling, validation loss rising → overfitting. Regularise, or get more data.
   - Jagged or `nan` loss → the learning rate is too high.
   - Nothing moves at all → check input scaling, the learning rate, and that your hidden layers have activations.
7. **Change one thing, retrain, overlay the curves.** Never two things at once.
8. **Touch the test set once**, at the very end.

### Output layer cheat sheet

| Task | Last layer | Loss | Label format |
|---|---|---|---|
| Regression | `Dense(1)` | `mse` or `mae` | float |
| Binary | `Dense(1, "sigmoid")` | `binary_crossentropy` | 0 / 1 |
| Multi-class | `Dense(k, "softmax")` | `sparse_categorical_crossentropy` | integer 0…k−1 |
| Multi-class, one-hot | `Dense(k, "softmax")` | `categorical_crossentropy` | one-hot vector |
| Multi-label | `Dense(k, "sigmoid")` | `binary_crossentropy` | multi-hot vector |

### Exercises

1. Rerun Part 3 with `noise=0.05` in `make_moons`. Does the deepest network still overfit?
2. In Part 4, raise the depth to six hidden layers and compare `sigmoid` with `relu` again. Quantify the vanishing-gradient effect.
3. Write a callback that stops training as soon as validation accuracy passes 0.95.
4. Replace the MNIST dense stack with `Conv2D` + `MaxPooling2D` layers (remember to reshape the input to `(28, 28, 1)`), and compare parameter count and accuracy against the best dense model here.
5. Turn the digit classifier into a regression problem — predict the digit as a number with a linear output and `mse`. Look at the predictions and explain why this is the wrong framing.